In [45]:
# Importations

import time
from enderscope import SerialUtils, Stage 
import serial
from math import *
import threading

In [ ]:
# Variables

  # Variables modifiables

rectangle = [40,30] # en mm, x puis y
coordinate_rec_one = [10,10] # coordonnées origine [gauche, bas] du carré lié à la position de la boîte de pétri
printer_speed = 30
total_thickness = 2
location_purge_A = [150, 150,8]


  #  Variables fixes
width_extrusion = 5 # en mm (width_extrusion x épaisseur = V_seringue x Surface seringue / V_imprimante)
number_passages = min(ceil(rectangle[0] /(2 * width_extrusion)),ceil(rectangle[1] /(2 * width_extrusion)))  # arrondit à l'entier superieur 
thickness_1_layer =  2
number_layer = ceil(total_thickness/thickness_1_layer)
coordinate_rec_2 = [coordinate_rec_one[0]+rectangle[0], coordinate_rec_one[1]] # Coordoones de l origine pour le function_rectangle 2
coordinate_rec_3 = [coordinate_rec_one[0]+ 2 *rectangle[0], coordinate_rec_one[1]]

gap = [1,1] # Pour le pousse seringue A, décalage de 1 selon x et 1 selon y. Si centré, on en deduit les gaps des autres 

height_printert = 10 # en mm, dépend en partie de la hauteur de la boîte de pétri

In [ ]:
# Ports

ports = SerialUtils.serial_ports() # Liste des ports
print (ports) # Affiche la liste

syringe_pump_port = ports[0] # A modifier en fonction du branchement
printer_port = ports[1] # A modifier en fonction du branchement

s = Stage(printer_port, 115200) # Connexion imprimante
syringe_pump = serial.Serial(port= syringe_pump_port, baudrate=115200, timeout=0.01, writeTimeout=1) # Connexion pousse seringue

['COM6', 'COM9']


In [ ]:
# Message respectif à envoyer à un pousse seringue pour le débloquer

def message_start(letter_syringe_pump): #
    return(f"{letter_syringe_pump}\n".encode('utf8')) # on transforme la f string en bytes avec la commande de fin encode

In [ ]:
# Message reculer
def message_back_off():
    return b"R\n"

In [ ]:
# Message respectif STOP

def message_stop():
        return b"S\n"

In [ ]:
#Purge 

def purge_syringe_pump(letter_syringe_pump):
    syringe_pump.write(message_start(letter_syringe_pump))
    time.sleep(2)
    syringe_pump.write(message_reculer())
    time.sleep(2)
    syringe_pump.write(message_stop())
    time.sleep(1)

purge_A = threading.Thread( target = purge_syringe_pump, args = ('A'))

def purge():
    s.move_absolute(location_purge_A[0],location_purge_A[1],location_purge_A[2])
    s.write_code(f"M400")
    purge_A.start()
    purge_A.join()

In [ ]:
# Déterminer la postition qui'il faut demadnder à l'imprimante en fonction du pousse seringue utilisé

def position(letter_syringe_pump, coordonnees):
    position_tube = []
    if letter_syringe_pump == "A":
        position_tube.append(coordonnees[0] - gap[0])
        position_tube.append(coordonnees[1] - gap[1])
        return position_tube
    
    if letter_syringe_pump == "B":
        position_tube.append(coordonnees[0] + gap[0])
        position_tube.append(coordonnees[1] - gap[1])
        return position_tube

    if letter_syringe_pump == "C":
        position_tube.append(coordonnees[0] + gap[0])
        position_tube.append(coordonnees[1] + gap[1])
        return position_tube

In [ ]:
# Forme un carré

def function_rectangle(): # Position du coin (bas_gauche) du function_rectangle

    x_rectangle = rectangle[0]
    y_rectangle = rectangle[1]

    for i in range (number_passages):
        for j in range(number_layer):
            s.write_code(f"M203 X{printer_speed}")
            s.write_code(f"M203 Y{printer_speed}")
            s.move_axis('x', x_rectangle)
            s.move_axis ('y', y_rectangle)
            s.move_axis('x', -x_rectangle)
            s.move_axis('y', - (y_rectangle -width_extrusion))

    

            s.move_axis('x', width_extrusion)
        
            x_rectangle = x_rectangle - (2 * width_extrusion)
            y_rectangle = y_rectangle - (2*width_extrusion)

    s.write_code(f"M400")

In [ ]:
def go (letter_syringe_pump, coordonnes_debut):
    s.move_absolute(position(letter_syringe_pump, coordonnes_debut)[0],position(letter_syringe_pump, coordonnes_debut)[1], height_printert)    
    s.write_code(f"M400") 

In [ ]:
# On fait le focus

s.home()

 #On fait la purge
purge()



# Tracer le premier carré avec le pousse seringue A
go('A', coordinate_rec_one)  # Aller en bas a gauche du function_rectangle 
syringe_pump.write(message_start('A'))
function_rectangle() 
syringe_pump.write(message_stop())

# Tracer le deuxième carré avec le pousse seringue B
go('A', coordinate_rec_2)
syringe_pump.write(message_start('A'))
function_rectangle() 
syringe_pump.write(message_stop())

# Tracer le troisième carré avec le pousse seringue C
go('A', coordinate_rec_3)
syringe_pump.write(message_start('A'))
function_rectangle() 
syringe_pump.write(message_stop())



SerialException: GetOverlappedResult failed (PermissionError(13, 'Accès refusé.', None, 5))